<a href="https://colab.research.google.com/github/vitormankowskicorrea-lab/Programa-oParaCi-nciaDeDados/blob/main/analise_steam_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Análise de Oportunidades e Estrutura de Preços na Plataforma Steam

Trabalho individual de Programação para Ciência de Dados (arquivos, Pandas, NumPy, POO e Web Scraping).

Execute as células em ordem. Por padrão o notebook lê o arquivo steam_promocoes.csv (se ele não existir, é criado automaticamente). Para tentar coletar dados reais da loja, troque USAR_SCRAPING = True na seção 2.

## 1. Definição do Problema

**Tema:** Análise de Oportunidades e Estrutura de Preços na Plataforma Steam.

**Problema:** como se comportam os preços originais, os percentuais de desconto e os preços finais dos jogos em promoção? É viável filtrar as melhores oportunidades por faixa de preço?

**O que queremos descobrir:**
- como os descontos se distribuem (média, mediana, dispersão);
- a média de preços por gênero e por status de oferta (com ou sem desconto);
- se existem preços atípicos (outliers);
- um ranking dos jogos com maior desconto e maior economia;
- se jogos mais caros têm descontos maiores.

## 2. Obtenção e Exploração dos Dados

Há duas formas de obter os dados:

1. coletar_via_scraping() lê a página de ofertas da Steam com requests e BeautifulSoup.
2. gerar_csv() cria um CSV simulado com o mesmo formato de texto da loja (R$ 59,99, -50%), incluindo valores ausentes e duplicatas para praticar a limpeza.

**Atenção:** os valores do CSV são simulados, não são preços reais da Steam.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup

ARQUIVO = "steam_promocoes.csv"
USAR_SCRAPING = False

In [ ]:
def coletar_via_scraping():
    url = "https://store.steampowered.com/search/?specials=1&cc=br&l=brazilian"
    resposta = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    sopa = BeautifulSoup(resposta.text, "html.parser")

    def texto(item, seletor):
        achado = item.select_one(seletor)
        return achado.get_text(strip=True) if achado else None

    linhas = []
    for item in sopa.select("a.search_result_row"):
        linhas.append({
            "titulo": texto(item, "span.title"),
            "genero": None,
            "preco_original": texto(item, "div.discount_original_price"),
            "desconto": texto(item, "div.discount_pct"),
            "preco_final": texto(item, "div.discount_final_price"),
            "avaliacao": None,
        })
    return pd.DataFrame(linhas)

In [ ]:
def gerar_csv(n=600):
    np.random.seed(42)
    nomes = ["Shadow Realm", "Iron Legacy", "Neon Siege", "Lost Kingdom", "Pixel Quest",
             "Crimson Arena", "Frozen Odyssey", "Wild Frontier", "Dark Tactics", "Golden Harvest"]
    generos = ["Ação", "RPG", "Estratégia", "Aventura", "Simulação", "Indie", "Corrida", "Esportes"]
    descontos = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
    prob_desc = [0.18, 0.07, 0.09, 0.10, 0.11, 0.14, 0.10, 0.09, 0.07, 0.05]

    linhas = []
    for i in range(n):
        preco = int(np.clip(np.random.lognormal(4, 0.7), 8, 450)) + 0.99
        desconto = np.random.choice(descontos, p=prob_desc)
        final = round(preco * (1 - desconto / 100), 2)
        gratis = i < 15   # os 15 primeiros jogos são gratuitos
        linhas.append({
            "titulo": f"{np.random.choice(nomes)} {i + 1}",
            "genero": np.random.choice(generos),
            "preco_original": "Grátis" if gratis else f"R$ {preco:.2f}".replace(".", ","),
            "desconto": f"-{desconto}%" if desconto > 0 and not gratis else np.nan,
            "preco_final": "Grátis" if gratis else f"R$ {final:.2f}".replace(".", ","),
            "avaliacao": f"{int(np.clip(np.random.normal(78, 12), 35, 99))}%",
        })
    df = pd.DataFrame(linhas)

    # problemas propositais: ausentes e duplicatas
    df.loc[df.sample(20).index, "genero"] = np.nan
    df.loc[df.sample(40).index, "avaliacao"] = np.nan
    df.loc[df[df["desconto"].notna()].sample(12).index, "preco_final"] = np.nan
    df = pd.concat([df, df.sample(15)]).sample(frac=1).reset_index(drop=True)

    df.to_csv(ARQUIVO, index=False)
    return df

In [ ]:
if USAR_SCRAPING:
    df_bruto = coletar_via_scraping()
    df_bruto.to_csv(ARQUIVO, index=False)
elif os.path.exists(ARQUIVO):
    df_bruto = pd.read_csv(ARQUIVO)
else:
    df_bruto = gerar_csv()

df_bruto.head(10)

,titulo,genero,preco_original,desconto,preco_final,avaliacao
0,Frozen Odyssey 394,Ação,"R$ 57,99",-30%,"R$ 40,59",93%
1,Golden Harvest 381,RPG,"R$ 134,99",NaN,"R$ 134,99",86%
2,Pixel Quest 205,Indie,"R$ 73,99",-10%,NaN,80%
3,Lost Kingdom 406,Esportes,"R$ 155,99",NaN,"R$ 155,99",74%
4,Lost Kingdom 39,Aventura,"R$ 35,99",-90%,"R$ 3,60",71%
5,Golden Harvest 38,Estratégia,"R$ 31,99",-80%,"R$ 6,40",74%
6,Crimson Arena 275,Aventura,"R$ 33,99",-30%,"R$ 23,79",76%
7,Iron Legacy 529,Esportes,"R$ 53,99",-20%,"R$ 43,19",56%
8,Dark Tactics 481,Corrida,"R$ 95,99",-70%,"R$ 28,80",75%
9,Crimson Arena 231,NaN,"R$ 77,99",-50%,"R$ 38,99",88%


### 2.1 Dimensões e tipos de dados

In [ ]:
print("Linhas e colunas:", df_bruto.shape)
df_bruto.info()

Linhas e colunas: (615, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 615 entries, 0 to 614
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   titulo          615 non-null    object
 1   genero          593 non-null    object
 2   preco_original  615 non-null    object
 3   desconto        497 non-null    object
 4   preco_final     603 non-null    object
 5   avaliacao       574 non-null    object
dtypes: object(6)
memory usage: 29.0+ KB


### 2.2 Valores ausentes, duplicatas e inconsistências

In [ ]:
print("Valores ausentes:")
print(df_bruto.isna().sum())
print()
print("Linhas duplicadas:", df_bruto.duplicated().sum())
print("Preços originais sem 'R$':", df_bruto.loc[~df_bruto["preco_original"].str.contains("R$", regex=False), "preco_original"].unique())

Valores ausentes:
titulo              0
genero             22
preco_original      0
desconto          118
preco_final        12
avaliacao          41
dtype: int64

Linhas duplicadas: 15
Preços originais sem 'R$': ['Grátis']


**Problemas encontrados:**
- Preço, desconto e avaliação vieram como texto (`object`) por causa de `R$`, `%` e do sinal `-`.
- A vírgula é o separador decimal, então `float("59,99")` dá erro.
- Há valores ausentes em `genero`, `avaliacao` e `preco_final`.
- `desconto` vazio significa jogo sem promoção (não é erro de coleta).
- Existem linhas duplicadas.
- Jogos gratuitos aparecem como "Grátis" em vez de número.

## 3. Preparação dos Dados

Cada passo vem com a justificativa no comentário.

In [ ]:
def limpar_preco(texto):
    if pd.isna(texto):
        return np.nan
    if "R$" not in texto:          # "Grátis" vira 0
        return 0.0
    return float(texto.replace("R$", "").replace(",", ".").strip())

def limpar_percentual(texto):
    if pd.isna(texto):
        return np.nan
    return float(texto.replace("-", "").replace("%", ""))

df = df_bruto.drop_duplicates().reset_index(drop=True)   # duplicatas distorcem médias e contagens

df["preco_original"] = df["preco_original"].apply(limpar_preco)
df["preco_final"] = df["preco_final"].apply(limpar_preco)
df["desconto"] = df["desconto"].apply(limpar_percentual)
df["avaliacao"] = df["avaliacao"].apply(limpar_percentual)

df["desconto"] = df["desconto"].fillna(0)   # sem desconto informado = 0%

# preço final ausente pode ser recalculado com os outros dois valores
df["preco_final"] = df["preco_final"].fillna(df["preco_original"] * (1 - df["desconto"] / 100)).round(2)

df["genero"] = df["genero"].fillna("Não informado")   # mantém a linha, o preço dela ainda é útil
df["avaliacao"] = df["avaliacao"].fillna(df["avaliacao"].median())   # mediana é menos sensível a extremos

df["economia"] = df["preco_original"] - df["preco_final"]
df["status_oferta"] = np.where(df["desconto"] > 0, "Em promoção", "Sem desconto")

# jogos gratuitos não têm desconto e puxariam as médias para baixo
print("Jogos gratuitos removidos da análise:", (df["preco_original"] == 0).sum())
df_pago = df[df["preco_original"] > 0].reset_index(drop=True)

print("Formato final:", df_pago.shape)
print("Valores ausentes restantes:", df_pago.isna().sum().sum())
df_pago.info()

Jogos gratuitos removidos da análise: 15
Formato final: (585, 8)
Valores ausentes restantes: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 585 entries, 0 to 584
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   titulo          585 non-null    object 
 1   genero          585 non-null    object 
 2   preco_original  585 non-null    float64
 3   desconto        585 non-null    float64
 4   preco_final     585 non-null    float64
 5   avaliacao       585 non-null    float64
 6   economia        585 non-null    float64
 7   status_oferta   585 non-null    object 
dtypes: float64(5), object(3)
memory usage: 36.7+ KB


In [ ]:
df_pago.head()

,titulo,genero,preco_original,desconto,preco_final,avaliacao,economia,status_oferta
0,Frozen Odyssey 394,Ação,57.99,30.0,40.59,93.0,17.40,Em promoção
1,Golden Harvest 381,RPG,134.99,0.0,134.99,86.0,0.00,Sem desconto
2,Pixel Quest 205,Indie,73.99,10.0,66.59,80.0,7.40,Em promoção
3,Lost Kingdom 406,Esportes,155.99,0.0,155.99,74.0,0.00,Sem desconto
4,Lost Kingdom 39,Aventura,35.99,90.0,3.60,71.0,32.39,Em promoção


## 4. Análise dos Dados

### 4.1 Pandas

In [ ]:
df_pago[["preco_original", "desconto", "preco_final", "economia"]].describe()

,preco_original,desconto,preco_final,economia
count,585.000000,585.000000,585.000000,585.000000
mean,70.413932,40.871795,41.792974,28.620957
std,52.936329,27.624419,40.475215,31.349833
min,8.990000,0.000000,1.200000,0.000000
25%,35.990000,20.000000,17.100000,6.800000
50%,54.990000,40.000000,28.990000,19.790000
75%,88.990000,60.000000,50.990000,41.390000
max,370.990000,90.000000,264.990000,205.790000


In [ ]:
por_status = df_pago.groupby("status_oferta").agg(
    jogos=("titulo", "count"),
    preco_original_medio=("preco_original", "mean"),
    preco_final_medio=("preco_final", "mean"),
    preco_final_mediano=("preco_final", "median"),
    desvio_padrao=("preco_final", "std"),
)
por_status

,jogos,preco_original_medio,preco_final_medio,preco_final_mediano,desvio_padrao
status_oferta,,,,,
Em promoção,486,68.932387,34.481235,25.50,32.237751
Sem desconto,99,77.686970,77.686970,58.99,55.250349


In [ ]:
promo = df_pago[df_pago["status_oferta"] == "Em promoção"].copy()

por_genero = promo.groupby("genero").agg(
    jogos=("titulo", "count"),
    desconto_medio=("desconto", "mean"),
    preco_final_medio=("preco_final", "mean"),
).sort_values("desconto_medio", ascending=False)
por_genero

,jogos,desconto_medio,preco_final_medio
genero,,,
RPG,55,55.090909,26.313636
Não informado,17,54.705882,29.754706
Estratégia,61,51.311475,39.815082
Aventura,58,51.034483,35.233966
Simulação,69,49.130435,32.232899
Corrida,52,47.307692,34.520000
Esportes,58,46.896552,32.291207
Indie,54,45.925926,41.817407
Ação,62,45.322581,35.199516


In [ ]:
promo["faixa"] = pd.cut(promo["preco_final"],
                        bins=[0, 20, 50, 100, 200, 500],
                        labels=["Até R$ 20", "R$ 20 a 50", "R$ 50 a 100", "R$ 100 a 200", "Acima de R$ 200"])

por_faixa = promo.groupby("faixa", observed=True).agg(
    jogos=("titulo", "count"),
    desconto_medio=("desconto", "mean"),
    economia_media=("economia", "mean"),
    preco_final_medio=("preco_final", "mean"),
)
por_faixa

,jogos,desconto_medio,economia_media,preco_final_medio
faixa,,,,
Até R$ 20,183,63.606557,27.068415,11.916120
R$ 20 a 50,209,42.535885,32.162440,32.042871
R$ 50 a 100,74,37.567568,49.190811,66.650541
R$ 100 a 200,17,30.000000,65.657059,131.450588
Acima de R$ 200,3,30.000000,103.833333,237.823333


In [ ]:
print("Top 10 maiores descontos:")
promo.sort_values(["desconto", "preco_final"], ascending=[False, True])[["titulo", "genero", "preco_original", "desconto", "preco_final"]].head(10)

Top 10 maiores descontos:


,titulo,genero,preco_original,desconto,preco_final
486,Lost Kingdom 445,Ação,11.99,90.0,1.2
371,Pixel Quest 503,RPG,18.99,90.0,1.9
166,Pixel Quest 217,Simulação,28.99,90.0,2.9
71,Neon Siege 514,Estratégia,29.99,90.0,3.0
557,Dark Tactics 328,Corrida,30.99,90.0,3.1
439,Lost Kingdom 366,Corrida,31.99,90.0,3.2
42,Shadow Realm 29,RPG,32.99,90.0,3.3
365,Iron Legacy 369,Estratégia,33.99,90.0,3.4
421,Frozen Odyssey 596,Corrida,34.99,90.0,3.5
4,Lost Kingdom 39,Aventura,35.99,90.0,3.6


In [ ]:
print("Top 10 maiores economias em reais:")
promo.sort_values("economia", ascending=False)[["titulo", "genero", "preco_original", "desconto", "preco_final", "economia"]].head(10)

Top 10 maiores economias em reais:


,titulo,genero,preco_original,desconto,preco_final,economia
241,Wild Frontier 292,Ação,293.99,70.0,88.20,205.79
75,Dark Tactics 281,Esportes,252.99,80.0,50.60,202.39
393,Pixel Quest 196,Esportes,234.99,80.0,47.00,187.99
458,Neon Siege 307,Esportes,229.99,80.0,46.00,183.99
228,Crimson Arena 304,Indie,278.99,60.0,111.60,167.39
406,Pixel Quest 431,Estratégia,191.99,80.0,38.40,153.59
143,Pixel Quest 49,Aventura,370.99,40.0,222.59,148.40
182,Lost Kingdom 62,Esportes,290.99,50.0,145.50,145.49
429,Iron Legacy 260,Aventura,194.99,70.0,58.50,136.49
116,Iron Legacy 136,Corrida,191.99,70.0,57.60,134.39


### 4.2 NumPy

In [ ]:
descontos = promo["desconto"].to_numpy()
originais = promo["preco_original"].to_numpy()
precos = df_pago["preco_final"].to_numpy()

print("Desconto médio:", round(np.mean(descontos), 2))
print("Desconto mediano:", np.median(descontos))
print("Desvio padrão:", round(np.std(descontos, ddof=1), 2))
print("Percentis 25, 50, 75 e 90:", np.percentile(descontos, [25, 50, 75, 90]))

limiar = np.percentile(descontos, 75)
print(f"\nJogos com desconto igual ou maior que {limiar:.0f}% (top 25%):", np.sum(descontos >= limiar))

q1, q3 = np.percentile(precos, [25, 75])
limite_superior = q3 + 1.5 * (q3 - q1)
print(f"\nLimite de outlier no preço final (método IQR): R$ {limite_superior:.2f}")
print("Jogos acima desse limite:", np.sum(precos > limite_superior))

correlacao = np.corrcoef(originais, descontos)[0, 1]
print("\nCorrelação entre preço original e desconto:", round(correlacao, 3))

Desconto médio: 49.2
Desconto mediano: 50.0
Desvio padrão: 22.55
Percentis 25, 50, 75 e 90: [30. 50. 70. 80.]

Jogos com desconto igual ou maior que 70% (top 25%): 134

Limite de outlier no preço final (método IQR): R$ 101.83
Jogos acima desse limite: 48

Correlação entre preço original e desconto: 0.046


### 4.3 Programação Orientada a Objetos

A classe AnalisadorDePromocoes guarda o DataFrame e o limite de desconto considerado "bom".
- _df é protegido (um sublinhado).
- __limite_desconto é privado (dois sublinhados).
- @property e @limite_desconto.setter funcionam como getter e setter, e o setter valida o valor.

In [ ]:
class AnalisadorDePromocoes:
    def __init__(self, dataframe, limite_desconto=50):
        self._df = dataframe.copy()
        self.limite_desconto = limite_desconto

    @property
    def limite_desconto(self):
        return self.__limite_desconto

    @limite_desconto.setter
    def limite_desconto(self, valor):
        if valor < 0 or valor > 100:
            raise ValueError("O limite deve estar entre 0 e 100.")
        self.__limite_desconto = valor

    def resumo(self):
        promo = self._df[self._df["desconto"] > 0]
        return {
            "total_jogos": len(self._df),
            "jogos_em_promocao": len(promo),
            "desconto_medio": round(promo["desconto"].mean(), 2),
            "desconto_mediano": round(promo["desconto"].median(), 2),
            "preco_final_mediano": round(promo["preco_final"].median(), 2),
        }

    def melhores_oportunidades(self, n=10, preco_max=None):
        filtro = self._df[self._df["desconto"] >= self.__limite_desconto]
        if preco_max is not None:
            filtro = filtro[filtro["preco_final"] <= preco_max]
        filtro = filtro.sort_values(["desconto", "avaliacao"], ascending=False)
        return filtro[["titulo", "genero", "preco_original", "desconto", "preco_final", "avaliacao"]].head(n)

    def outliers_preco(self):
        precos = self._df["preco_final"].to_numpy()
        q1, q3 = np.percentile(precos, [25, 75])
        limite = q3 + 1.5 * (q3 - q1)
        return self._df[self._df["preco_final"] > limite].sort_values("preco_final", ascending=False)

In [ ]:
analisador = AnalisadorDePromocoes(df_pago, limite_desconto=50)

for chave, valor in analisador.resumo().items():
    print(chave, ":", valor)

try:
    analisador.limite_desconto = 150
except ValueError as erro:
    print("\nErro esperado:", erro)

total_jogos : 585
jogos_em_promocao : 486
desconto_medio : 49.2
desconto_mediano : 50.0
preco_final_mediano : 25.5

Erro esperado: O limite deve estar entre 0 e 100.


In [ ]:
print("Desconto de 50% ou mais e preço final até R$ 40:")
analisador.melhores_oportunidades(n=10, preco_max=40)

Desconto de 50% ou mais e preço final até R$ 40:


,titulo,genero,preco_original,desconto,preco_final,avaliacao
398,Frozen Odyssey 428,Simulação,55.99,90.0,5.6,94.0
63,Pixel Quest 393,Corrida,115.99,90.0,11.6,88.0
486,Lost Kingdom 445,Ação,11.99,90.0,1.2,88.0
365,Iron Legacy 369,Estratégia,33.99,90.0,3.4,86.0
324,Golden Harvest 74,Estratégia,49.99,90.0,5.0,85.0
156,Lost Kingdom 345,RPG,56.99,90.0,5.7,84.0
506,Frozen Odyssey 573,Não informado,55.99,90.0,5.6,84.0
42,Shadow Realm 29,RPG,32.99,90.0,3.3,83.0
452,Wild Frontier 185,Ação,42.99,90.0,4.3,83.0
269,Crimson Arena 558,Ação,55.99,90.0,5.6,80.0


In [ ]:
print("Outliers de preço final:")
analisador.outliers_preco()[["titulo", "genero", "preco_original", "desconto", "preco_final"]].head(8)

Outliers de preço final:


,titulo,genero,preco_original,desconto,preco_final
26,Neon Siege 581,Estratégia,264.99,0.0,264.99
171,Pixel Quest 177,Estratégia,330.99,20.0,264.79
467,Golden Harvest 390,Corrida,257.99,0.0,257.99
278,Dark Tactics 579,Indie,237.99,0.0,237.99
281,Golden Harvest 119,Não informado,230.99,0.0,230.99
118,Dark Tactics 163,Simulação,322.99,30.0,226.09
143,Pixel Quest 49,Aventura,370.99,40.0,222.59
494,Golden Harvest 310,Simulação,198.99,0.0,198.99


### 4.5 Respostas às perguntas do problema

| Pergunta | Onde olhar |
|---|---|
| Como os descontos se distribuem? | `describe()`, bloco NumPy e histograma |
| Qual a média de preço por gênero e por status? | tabelas `por_status` e `por_genero` |
| Existem preços atípicos? | método IQR e `outliers_preco()` |
| Vale filtrar por faixa de preço? | tabela `por_faixa` e `melhores_oportunidades()` |
| Preço alto implica desconto maior? | correlação com `np.corrcoef` |

A interpretação dos números está na conclusão.

## 5. Documentação e Conclusões

**Resumo das etapas:**
1. *Obtenção:* scraping com requests e BeautifulSoup implementado, com CSV simulado como alternativa.
2. *Exploração:* .shape, .info()`, .isna().sum() e .duplicated() mostraram números guardados como texto, valores ausentes e duplicatas.
3. *Preparação:* duplicatas removidas, R$ e % retirados, preço final ausente recalculado, gênero ausente marcado como "Não informado", avaliação ausente preenchida com a mediana e jogos gratuitos separados.
4. *Análise:* Pandas (groupby, sort_values, pd.cut`), NumPy (percentis, IQR, correlação) e uma classe para organizar as consultas.

### Conclusão

Dos 585 jogos pagos da base, 486 (cerca de 83%) estavam em promoção. Entre eles, o desconto médio foi de 49,2% e a mediana de 50%, com desvio padrão de 22,6 pontos. Cerca de 57% dos jogos em promoção têm 50% ou mais de desconto, e 25% deles chegam a 70% ou mais. O preço final mediano em oferta foi de R$ 25,50, contra R$ 58,99 dos jogos sem desconto. A média dos sem desconto também é bem maior (R$ 77,69 contra R$ 34,48), o que indica que, nesta base, os jogos mais caros são os que menos entram em promoção.

Por gênero, o desconto médio vai de 45,3% (Ação) a 55,1% (RPG), uma diferença de cerca de dez pontos. O gênero ajuda um pouco, mas não decide se vale a pena comprar. A correlação entre preço original e percentual de desconto foi de 0,046, praticamente zero, então um jogo mais caro não costuma ter desconto maior por causa do preço.

Nas faixas de preço, o desconto médio cai de 63,6% (até R$ 20) para 30% (acima de R$ 100), mas a economia em reais sobe de R$ 27 para mais de R$ 100. Um cuidado: as faixas foram feitas pelo preço final, e um desconto alto já baixa o preço final, então parte desse padrão vem da própria construção. Mesmo assim, filtrar por faixa e por limiar de desconto é viável, e a classe AnalisadorDePromocoes faz isso. Por exemplo, ela lista jogos com 50% ou mais de desconto e preço final até R$ 40, ordenados também pela avaliação.

Sobre preços atípicos, 48 jogos ficam acima de R$ 101,83 pelo critério do IQR. Muitos deles não têm desconto nenhum.

**Limitação:** os dados são simulados para imitar o formato da Steam. Os números mostram o método funcionando, mas não descrevem o mercado real. Se você rodar com dados reais (USAR_SCRAPING = True` ou outro CSV), o código funciona igual, mas esta conclusão precisa ser reescrita com os novos resultados.